# Tăng cường đặc trưng VSL

Tạo bản nhiễu từ mẫu gốc trong `data/processed`; `data/raw` giữ nguyên. Đặt `MIN_SAMPLES_PER_CLASS = N` để chỉ bổ sung các lớp có ít hơn N mẫu hoàn chỉnh trong `processed`, tạo đúng số còn thiếu. Đặt `-1` để tăng cường tất cả mẫu gốc theo `COPIES_PER_SAMPLE` mà không giới hạn. Mỗi bản mới ghi `origin_sample_id` để đi cùng fold với mẫu gốc. Validation trong `train.ipynb` chỉ chấm mẫu gốc.


In [5]:
import importlib, sys
from pathlib import Path
import numpy as np
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').is_dir()), None)
if PROJECT_ROOT is None: raise FileNotFoundError('Chạy Jupyter từ dự án hoặc notebooks/')
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
reader = importlib.reload(importlib.import_module('src.data.data_reader'))
augment = importlib.reload(importlib.import_module('src.data.augment'))
DataHandler = reader.DataHandler
PROCESSED_ROOT = PROJECT_ROOT / 'data' / 'processed'
RAW_ROOT = PROJECT_ROOT / 'data' / 'raw'
SELECTED_CLASSES = 'all'  # 'all', 'xin chào', hoặc ['xin chào', 'cảm ơn']
MIN_SAMPLES_PER_CLASS = int(input("MIN_SAMPLES_PER_CLASS (-1 to all): "))  # N; -1 = tăng cường toàn bộ, không giới hạn
COPIES_PER_SAMPLE = 2    # chỉ dùng khi MIN_SAMPLES_PER_CLASS = -1
NOISE_STRENGTH = 1.0    # khoảng (0, 3]
SEED = None  # đặt số nguyên để chạy tái lập
if MIN_SAMPLES_PER_CLASS < -1 or COPIES_PER_SAMPLE < 1 or not 0 < NOISE_STRENGTH <= 3: raise ValueError('Cấu hình ngưỡng/số bản/noise không hợp lệ')
classes = None if SELECTED_CLASSES == 'all' else ([SELECTED_CLASSES] if isinstance(SELECTED_CLASSES, str) else SELECTED_CLASSES)
plan, counts = augment.plan_augmentation(PROCESSED_ROOT, classes, MIN_SAMPLES_PER_CLASS, COPIES_PER_SAMPLE)
print(f'{sum(copies for _, copies in plan)} mẫu dự kiến tạo trong {len(counts)} lớp')


880 mẫu dự kiến tạo trong 16 lớp


## Kiểm tra trước khi tạo


In [6]:
for name, count in counts.items():
    extra = sum(copies for source, copies in plan if source.parent.name == name)
    print(f'{name}: hiện có {count}, thêm {extra}' + (f', sau đó có {count + extra}' if MIN_SAMPLES_PER_CLASS != -1 else ''))
print('Đã đủ ngưỡng, không cần tạo thêm.' if not plan else 'Raw và mẫu gốc được giữ nguyên. Chạy ô tiếp theo để tạo bản nhiễu.')


age: hiện có 45, thêm 55, sau đó có 100
cảm ơn: hiện có 45, thêm 55, sau đó có 100
dấu nặng: hiện có 15, thêm 85, sau đó có 100
g: hiện có 45, thêm 55, sau đó có 100
h: hiện có 60, thêm 40, sau đó có 100
i: hiện có 45, thêm 55, sau đó có 100
k: hiện có 45, thêm 55, sau đó có 100
m: hiện có 60, thêm 40, sau đó có 100
n: hiện có 45, thêm 55, sau đó có 100
r: hiện có 45, thêm 55, sau đó có 100
t: hiện có 45, thêm 55, sau đó có 100
ten_la: hiện có 45, thêm 55, sau đó có 100
toi: hiện có 45, thêm 55, sau đó có 100
u: hiện có 45, thêm 55, sau đó có 100
xin chào: hiện có 45, thêm 55, sau đó có 100
địa chỉ: hiện có 45, thêm 55, sau đó có 100
Raw và mẫu gốc được giữ nguyên. Chạy ô tiếp theo để tạo bản nhiễu.


## Tạo mẫu tăng cường
Ô này tính lại số mẫu hiện có ngay trước khi ghi. Với ngưỡng N, chạy lại sau khi đã đủ N sẽ không tạo thêm; với N = -1, mỗi lần chạy tạo thêm bản mới.


In [7]:
rng = np.random.default_rng(SEED)
run_plan, current_counts = augment.plan_augmentation(PROCESSED_ROOT, classes, MIN_SAMPLES_PER_CLASS, COPIES_PER_SAMPLE)
created, errors = [], []
for index, (source, copies) in enumerate(run_plan, 1):
    try:
        paths = augment.augment_sample(source, RAW_ROOT, copies, NOISE_STRENGTH, rng)
        created.extend(paths)
        print(f'[{index}/{len(run_plan)}] {source.parent.name}/{source.name} -> {", ".join(path.name for path in paths)}', flush=True)
    except Exception as exc:
        errors.append((str(source), str(exc)))
        print(f'[{index}/{len(run_plan)}] LỖI {source}: {exc}', flush=True)
print(f'Đã tạo {len(created)} mẫu; lỗi {len(errors)}')


[1/260] age/0 -> 45, 46, 47, 48
[2/260] age/1 -> 49, 50, 51, 52
[3/260] age/10 -> 53, 54, 55
[4/260] age/11 -> 56, 57, 58
[5/260] age/12 -> 59, 60, 61
[6/260] age/13 -> 62, 63, 64
[7/260] age/14 -> 65, 66, 67
[8/260] age/2 -> 68, 69, 70, 71
[9/260] age/3 -> 72, 73, 74, 75
[10/260] age/4 -> 76, 77, 78, 79
[11/260] age/5 -> 80, 81, 82, 83
[12/260] age/6 -> 84, 85, 86, 87
[13/260] age/7 -> 88, 89, 90, 91
[14/260] age/8 -> 92, 93, 94, 95
[15/260] age/9 -> 96, 97, 98, 99
[16/260] cảm ơn/0 -> 45, 46, 47, 48
[17/260] cảm ơn/1 -> 49, 50, 51, 52
[18/260] cảm ơn/10 -> 53, 54, 55
[19/260] cảm ơn/11 -> 56, 57, 58
[20/260] cảm ơn/12 -> 59, 60, 61
[21/260] cảm ơn/13 -> 62, 63, 64
[22/260] cảm ơn/14 -> 65, 66, 67
[23/260] cảm ơn/2 -> 68, 69, 70, 71
[24/260] cảm ơn/3 -> 72, 73, 74, 75
[25/260] cảm ơn/4 -> 76, 77, 78, 79
[26/260] cảm ơn/5 -> 80, 81, 82, 83
[27/260] cảm ơn/6 -> 84, 85, 86, 87
[28/260] cảm ơn/7 -> 88, 89, 90, 91
[29/260] cảm ơn/8 -> 92, 93, 94, 95
[30/260] cảm ơn/9 -> 96, 97, 98, 99
[31/

## Kiểm tra group


In [8]:
handler = DataHandler()
print('Mẫu đọc được:', handler.validate_dataset())
print('Tổng:', handler.get_class_counts())
print('Group gốc độc lập:', handler.get_origin_counts())
groups = np.asarray(handler.sample_groups('sample'))
augmented = np.asarray([handler.describe(i)['augmented'] for i in range(len(handler))])
assert all((~augmented[groups == group]).any() for group in np.unique(groups))
print(f'{augmented.sum()} bản tăng cường; mọi group đều có mẫu gốc')
print('Chạy lại train.ipynb từ ô đầu để dùng mẫu mới.')


Mẫu đọc được: 1600
Tổng: {'age': 100, 'cảm ơn': 100, 'dấu nặng': 100, 'g': 100, 'h': 100, 'i': 100, 'k': 100, 'm': 100, 'n': 100, 'r': 100, 't': 100, 'ten_la': 100, 'toi': 100, 'u': 100, 'xin chào': 100, 'địa chỉ': 100}
Group gốc độc lập: {'age': 15, 'cảm ơn': 15, 'dấu nặng': 15, 'g': 15, 'h': 30, 'i': 15, 'k': 15, 'm': 30, 'n': 15, 'r': 15, 't': 15, 'ten_la': 15, 'toi': 15, 'u': 15, 'xin chào': 15, 'địa chỉ': 15}
1330 bản tăng cường; mọi group đều có mẫu gốc
Chạy lại train.ipynb từ ô đầu để dùng mẫu mới.
